# M2 - Condition classification: train, evaluate, infer (Colab / Kaggle)

**Model:** Keras 3 / TensorFlow classifier `good | moderate | defective` (docs/01-prd.md §10.3).
**Plan:** docs/08-ml-plan.md §2/M2, §3.2, §5 - lightweight MobileNetV3-Small **baseline** vs
stronger EfficientNetV2-S **improved** on the *same* splits / augmentation / seed.

This notebook clones the repo and runs the existing scripts
(`ml/m2_condition_classification/scripts/`) end-to-end:
download data (Roboflow / Kaggle / upload) -> class-map -> 70/15/15 split -> train both
models -> evaluate on the frozen test split -> download the results.

**Before you start**
- Runtime -> Change runtime type -> **T4 GPU** (Colab) / Accelerator **GPU** (Kaggle). CPU works but is slow.
- Pick a license-permitted public damage/condition dataset (docs/08-ml-plan.md §3.2) and document URL + license in the report.
- In cell 1, set `REPO_URL` to your fork/repo.

Run cells in order. Skip the data cells you don't need (3A / 3B / 3C).


In [2]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# 1. SETUP - clone repo, point M2_WORK_ROOT at persistent storage
import os, pathlib

IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB = os.path.exists("/content")

# TODO: your fork/repo - the one edit that matters for cloning
REPO_URL = "https://github.com/rahulpandiyan/SafeResale.git"
BRANCH   = "main"

WORK_BASE = "/kaggle/working" if IS_KAGGLE else "/content"
REPO_DIR  = os.path.join(WORK_BASE, "SafeResale")

# Export REPO_DIR as an environment variable so it's accessible by os.environ
os.environ["REPO_DIR"] = REPO_DIR

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

# Data + runs live here (mirrors ~/safresale-ml/m2 in docs/10-setup-guide.md section 7)
os.environ["M2_WORK_ROOT"] = os.path.join(WORK_BASE, "safresale-ml", "m2")
pathlib.Path(os.environ["M2_WORK_ROOT"]).mkdir(parents=True, exist_ok=True)
print("M2_WORK_ROOT =", os.environ["M2_WORK_ROOT"])

%cd {REPO_DIR}/ml/m2_condition_classification


Cloning into '/kaggle/working/SafeResale'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 121 (delta 49), reused 90 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 99.79 KiB | 858.00 KiB/s, done.
Resolving deltas: 100% (49/49), done.
M2_WORK_ROOT = /kaggle/working/safresale-ml/m2
/kaggle/working/SafeResale/ml/m2_condition_classification


In [4]:
# 2. DEPS + GPU CHECK (TF/Keras already ship on Colab and Kaggle)
!pip install -q pyyaml matplotlib roboflow kaggle

import tensorflow as tf, keras
print("TF", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU devices:", [g.name for g in gpus] or "NONE - CPU fallback (much slower)")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.1 MB/s eta 0:00:00
TF 2.20.0 | Keras 3.13.2
GPU devices: ['/physical_device:GPU:0']


In [17]:
!pwd
!ls

/kaggle/working/SafeResale/ml/m2_condition_classification
configs  notebooks  README.md  scripts


In [5]:
%cd /kaggle/working/SafeResale
!git pull origin main

/kaggle/working/SafeResale
From https://github.com/rahulpandiyan/SafeResale
 * branch            main       -> FETCH_HEAD
Already up to date.


In [22]:
!nvidia-smi

Sun Aug 16 15:25:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Provide the dataset - run 3A AND 3B (combined), or 3C (your zip)

The model grades items as `good | moderate | defective`. No single free dataset has all
three grades, so we **combine two license-permitted sources** (configs/class_map.yaml):

| Option | Source | Feeds class | License |
|--------|--------|-------------|---------|
| 3A | Roboflow `damaged-vs-good-packages` (581 imgs, classes `damaged`/`intact`) | good + defective | CC BY 4.0 |
| 3B | Kaggle `dataclusterlabs/cracked-screen-dataset` (cracked screens) | moderate | CC0 (Public Domain) |
| 3C | Your own zip (upload / Google Drive) | any | yours |

**Run both 3A and 3B** - cell 4 merges them into one dataset. 3B is ~900 MB to download
(instant on Kaggle; slower on Colab) and needs a Kaggle account - on Colab it will ask
you to upload `kaggle.json` (https://www.kaggle.com/settings -> 'Create New API Token').

License caveat: many phone/electronics sets on Kaggle are **CC BY-NC** (non-commercial) -
avoid those for a marketplace. Note each dataset's URL + license in the report
(docs/08-ml-plan.md §3.2). Downloads land in `$M2_WORK_ROOT/data/datasets/<name>`.


In [6]:
# 3A. Roboflow Universe (classification) - damaged-vs-good-packages (CC BY 4.0)
#     Feeds: intact -> good, damaged -> defective. Free key: https://app.roboflow.com/settings/api
import os
ROBOFLOW_API_KEY = "QTXcA8ptjUufYfVJAc69"  # PASTE YOUR ROBOFLOW_API_KEY HERE
ROBOFLOW_WS   = "aadhavs-first-workspace"   # chosen dataset (damaged-vs-good-packages)
ROBOFLOW_PROJ = "damaged-vs-good-packages"  # 581 images, classes: damaged / intact
ROBOFLOW_VER  = 1

# Define the full path to the script to avoid issues with shell's current working directory
# REPO_DIR is defined in cell l4avFTJE8c-J
# Assuming `REPO_DIR` is '/kaggle/working/SafeResale' (from setup cell l4avFTJE8c-J)
# and the `download_dataset.py` script is inside 'ml/m2_condition_classification/scripts/'
script_path = os.path.join(os.environ["REPO_DIR"], "ml", "m2_condition_classification", "scripts", "download_dataset.py")

if ROBOFLOW_API_KEY and ROBOFLOW_PROJ != "mobile-phone-damage-detection":
    os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
    DATASET_NAME = "condition_packages"
    !python {script_path} --source roboflow --workspace {ROBOFLOW_WS} --project {ROBOFLOW_PROJ} --version {ROBOFLOW_VER} --name {DATASET_NAME}
else:
    print("Skipped - paste ROBOFLOW_API_KEY above, or use 3B / 3C.")

loading Roboflow workspace...
loading Roboflow project...

Extracting Dataset Version Zip to /kaggle/working/safresale-ml/m2/data/datasets/condition_packages in folder:: 100% 592/592 [00:00<00:00, 12606.38it/s]
top-level entries: ['test', 'train', 'valid']
  train: 412 images | damaged=183, intact=229
  valid: 112 images | damaged=51, intact=61
  test: 57 images | damaged=22, intact=35
Next: remap source labels in configs/class_map.yaml, then run prepare_dataset.py.


In [7]:
# 3B. Kaggle cracked-screen set (CC0 Public Domain - commercial OK) -> moderate
#     https://www.kaggle.com/datasets/dataclusterlabs/cracked-screen-dataset
#     Needs kaggle.json (auto on Kaggle; upload on Colab). ~900 MB download.
import os
import json # Needed for creating kaggle.json

IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB = os.path.exists("/content")

KAGGLE_DATASET = "dataclusterlabs/cracked-screen-dataset"

# PASTE YOUR KAGGLE USERNAME AND API KEY HERE to avoid interactive file upload
KAGGLE_USERNAME = "veereshkp" # e.g., "your_kaggle_username"
KAGGLE_KEY      = "KGAT_e5ab1d1ddaf5d06bab326e68f1b7264e" # e.g., "your_kaggle_api_key_xxxxxxxxxxxxxxxxxxxxxxxx"

kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")

if IS_COLAB:
    # Ensure the .kaggle directory exists
    os.makedirs(os.path.dirname(kaggle_json_path), exist_ok=True)

    if not os.path.exists(kaggle_json_path):
        if KAGGLE_USERNAME and KAGGLE_KEY:
            # Create kaggle.json programmatically from provided credentials
            kaggle_config = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
            with open(kaggle_json_path, "w") as f:
                json.dump(kaggle_config, f)
            os.chmod(kaggle_json_path, 0o600) # Set permissions
            print("kaggle.json created from provided KAGGLE_USERNAME and KAGGLE_KEY.")
        else:
            from google.colab import files
            print("Upload kaggle.json from https://www.kaggle.com/settings -> 'Create New API Token'")
            uploaded = files.upload()
            if "kaggle.json" in uploaded:
                with open(kaggle_json_path, "wb") as f:
                    f.write(uploaded["kaggle.json"])
                os.chmod(kaggle_json_path, 0o600)
                print("kaggle.json uploaded.")
            else:
                print("No kaggle.json uploaded. Please upload the file or provide credentials.")
    else:
        print("kaggle.json already exists.")

# Define the full path to the script to avoid issues with shell's current working directory
# REPO_DIR is defined in cell l4avFTJE8c-J
# Assuming `REPO_DIR` is '/kaggle/working/SafeResale' (from setup cell l4avFTJE8c-J)
# and the `download_dataset.py` script is inside 'ml/m2_condition_classification/scripts/'
script_path = os.path.join(os.environ["REPO_DIR"], "ml", "m2_condition_classification", "scripts", "download_dataset.py")

if KAGGLE_DATASET and os.path.exists(kaggle_json_path):
    DATASET_NAME = "condition_cracked"
    !python {script_path} --source kaggle --kaggle-dataset {KAGGLE_DATASET} --name {DATASET_NAME} --as-class cracked-screen
elif KAGGLE_DATASET:
    print("Skipped Kaggle dataset download - kaggle.json not found or not created. Please provide credentials or upload the file.")
else:
    print("Skipped - set KAGGLE_DATASET, or use 3A / 3C.")

kaggle.json created from provided KAGGLE_USERNAME and KAGGLE_KEY.
$ /usr/bin/python3 -m kaggle datasets download -d dataclusterlabs/cracked-screen-dataset -p /kaggle/working/safresale-ml/m2/data/datasets/condition_cracked/.download --unzip
/usr/bin/python3: No module named kaggle.__main__; 'kaggle' is a package and cannot be directly executed
Traceback (most recent call last):
  File "/kaggle/working/SafeResale/ml/m2_condition_classification/scripts/download_dataset.py", line 166, in <module>
    main()
  File "/kaggle/working/SafeResale/ml/m2_condition_classification/scripts/download_dataset.py", line 162, in main
    from_kaggle(args)
  File "/kaggle/working/SafeResale/ml/m2_condition_classification/scripts/download_dataset.py", line 126, in from_kaggle
    subprocess.run(cmd, check=True)
  File "/usr/lib/python3.12/subprocess.py", line 571, in run
    raise CalledProcessError(retcode, process.args,
subprocess.CalledProcessError: Command '['/usr/bin/python3', '-m', 'kaggle', 'dataset

In [8]:
# 3C. Upload your own zip (drag into the Files panel / upload dialog)
#     Expected inside the zip:  <class>/*.jpg   or   {train,valid,test}/<class>/*.jpg
import os, pathlib, zipfile
UPLOAD_ZIP = "condition_data.zip"   # TODO: name of the uploaded zip

if IS_COLAB and not os.path.exists(UPLOAD_ZIP):
    from google.colab import files
    files.upload()

if os.path.exists(UPLOAD_ZIP):
    DATASET_NAME = os.path.splitext(UPLOAD_ZIP)[0]
    dest = pathlib.Path(os.environ["M2_WORK_ROOT"]) / "data" / "datasets" / DATASET_NAME
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(UPLOAD_ZIP) as z:
        z.extractall(dest)
    print("extracted", len(list(dest.iterdir())), "top-level entries to", dest)
else:
    print("Skipped - no zip present; upload one or use 3A / 3B.")


KeyboardInterrupt: 

## 4. Map labels + build the split

`prepare_dataset.py` builds a **stratified 70/15/15** manifest; the test split is **frozen**
and never used in training. Imbalanced classes get inverse-frequency class weights
(no fabricated labels).

In [9]:
# 4. COMBINE SOURCES -> one dataset, map labels, then 70/15/15 split
#     (the test split is frozen and never used in training)
import os, pathlib, yaml

# Define the base directory for scripts and configs relative to REPO_DIR
m2_base_dir = pathlib.Path(os.environ["REPO_DIR"]) / "ml" / "m2_condition_classification"
script_path_combine = m2_base_dir / "scripts" / "combine_datasets.py"
yaml_path = m2_base_dir / "configs" / "class_map.yaml"
script_path_prepare = m2_base_dir / "scripts" / "prepare_dataset.py"

# Merge the two sources into a single class-folder dataset (condition_combined):
#   packages: intact -> good, damaged -> defective
#   cracked screens (CC0): -> moderate, capped at 1000 to keep classes balanced
!python {script_path_combine} --out condition_combined     --src condition_packages/intact     --src condition_packages/damaged     --src condition_cracked/cracked-screen:1000     --overwrite

# Class-map lives in configs/class_map.yaml (committed) - show it for the report
print("class map:", yaml.safe_load(yaml_path.read_text()))

DATASET_NAME = "condition_combined"
!python {script_path_prepare} --dataset {DATASET_NAME} --out {DATASET_NAME} --class-map {yaml_path.as_posix()}

condition_packages/intact                    found 325 images
  -> /kaggle/working/safresale-ml/m2/data/datasets/condition_combined/intact (325 images)
condition_packages/damaged                   found 256 images
  -> /kaggle/working/safresale-ml/m2/data/datasets/condition_combined/damaged (256 images)
skip condition_cracked/cracked-screen:1000: no images for class 'cracked-screen' in condition_cracked

combined dataset: /kaggle/working/safresale-ml/m2/data/datasets/condition_combined
  damaged          256
  intact           325

Next: map labels in configs/class_map.yaml, then run:
  python scripts/prepare_dataset.py --dataset condition_combined --out condition_combined --class-map configs/class_map.yaml
class map: {'good': ['intact'], 'moderate': ['cracked-screen'], 'defective': ['damaged']}
Per-class images (source):
  defective       256
  good            325

Split sizes (stratified 70/15/15):
  train    406  {'defective': 179, 'good': 227}
  val       86  {'defective': 38, 'goo

In [10]:
# 5. POINT configs/m2.yaml AT THIS MANIFEST
import os, pathlib, yaml

# Construct the full path to m2.yaml using REPO_DIR
m2_config_dir = pathlib.Path(os.environ["REPO_DIR"]) / "ml" / "m2_condition_classification" / "configs"
cfg_path = m2_config_dir / "m2.yaml"

cfg = yaml.safe_load(cfg_path.read_text())
cfg["dataset"] = DATASET_NAME
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, default_flow_style=False))
print("configs/m2.yaml dataset =", cfg["dataset"])
print("classes =", cfg["classes"], "| models =", [m["tag"] for m in cfg["models"]])

configs/m2.yaml dataset = condition_combined
classes = ['good', 'moderate', 'defective'] | models = ['baseline-m3small', 'improved-effv2s']


### Re-checking for `InvalidArgumentError` in images

Since the training still fails with `InvalidArgumentError` despite previous cleaning and manifest regeneration, let's explicitly check each image again using TensorFlow's own decoding functions. This will help identify any *new* or *remaining* exact image files that TensorFlow is struggling to decompress.

In [11]:
import tensorflow as tf
import os
import pathlib

M2_WORK_ROOT = os.environ.get("M2_WORK_ROOT")
DATASET_NAME = "condition_combined"
dataset_path = pathlib.Path(M2_WORK_ROOT) / "data" / "datasets" / DATASET_NAME

problematic_images = []

if dataset_path.exists():
    print(f"Re-checking all images in: {dataset_path} with TensorFlow's decoder...")
    for root, _, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg')):
                filepath = pathlib.Path(root) / file
                try:
                    # Read the image file content
                    img_bytes = tf.io.read_file(str(filepath))
                    # Attempt to decode it as a JPEG
                    _ = tf.image.decode_jpeg(img_bytes, channels=3)
                except tf.errors.InvalidArgumentError as e:
                    print(f"TensorFlow found problematic image: {filepath} - {e.message}")
                    problematic_images.append(filepath)
                except Exception as e:
                    print(f"An unexpected error occurred with {filepath}: {e}")

    if not problematic_images:
        print("No problematic JPEG images found by TensorFlow's decoder.")
        print("The `InvalidArgumentError` might be due to other factors or a transient issue within the data pipeline.")
    else:
        print(f"Found {len(problematic_images)} problematic images that TensorFlow cannot decode:")
        for img_path in problematic_images:
            print(f"  - {img_path}")
        print("You may need to manually inspect or remove these files, then re-run training.")
else:
    print(f"Dataset path not found: {dataset_path}")


Re-checking all images in: /kaggle/working/safresale-ml/m2/data/datasets/condition_combined with TensorFlow's decoder...
No problematic JPEG images found by TensorFlow's decoder.
The `InvalidArgumentError` might be due to other factors or a transient issue within the data pipeline.


### Removing problematic images from source dataset

Since the problematic images are being re-added by the `combine_datasets.py` script, we need to remove them from their original source location in the `condition_cracked` dataset.

In [12]:
import os
import pathlib

M2_WORK_ROOT = os.environ.get("M2_WORK_ROOT")

# Construct full paths to the problematic images in their original source dataset
problematic_images_in_source = [
    pathlib.Path(M2_WORK_ROOT) / "data" / "datasets" / "condition_cracked" / "cracked-screen" / "cracked-screen_Datacluster Cracked Screen (217).jpg",
    pathlib.Path(M2_WORK_ROOT) / "data" / "datasets" / "condition_cracked" / "cracked-screen" / "Datacluster Cracked Screen (217).jpg"
]

print("Attempting to remove problematic image files from original source dataset...")
for img_path in problematic_images_in_source:
    if img_path.exists():
        try:
            os.remove(img_path)
            print(f"Removed from source: {img_path}")
        except OSError as e:
            print(f"Error removing {img_path} from source: {e}")
    else:
        print(f"File not found in source, skipped: {img_path}")

print("Source image removal process completed.")
print("Please now re-run Cell 4 (to re-combine datasets without these files), then re-run Cell 6A and 6B (for training).")

Attempting to remove problematic image files from original source dataset...
File not found in source, skipped: /kaggle/working/safresale-ml/m2/data/datasets/condition_cracked/cracked-screen/cracked-screen_Datacluster Cracked Screen (217).jpg
File not found in source, skipped: /kaggle/working/safresale-ml/m2/data/datasets/condition_cracked/cracked-screen/Datacluster Cracked Screen (217).jpg
Source image removal process completed.
Please now re-run Cell 4 (to re-combine datasets without these files), then re-run Cell 6A and 6B (for training).


## 6. Train - same protocol for both models (seed 42, identical splits + augmentation)

Two-stage schedule per docs/08-ml-plan.md §5: head on frozen base, then full fine-tune at 1/10 LR.

In [13]:
# 6A. TRAIN BASELINE - MobileNetV3-Small (lightweight)
import os, pathlib

# Define the base directory for scripts and configs relative to REPO_DIR
m2_base_dir = pathlib.Path(os.environ["REPO_DIR"]) / "ml" / "m2_condition_classification"
train_script_path = m2_base_dir / "scripts" / "train.py"
m2_yaml_path = m2_base_dir / "configs" / "m2.yaml"

!python {train_script_path} --yaml {m2_yaml_path} --only baseline-m3small

2026-08-16 17:00:16.573140: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1786899616.574667    4052 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13653 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

=== baseline-m3small | MobileNetV3-Small | m2-condition-mobilenetv3small-v0.1 ===
train=406 val=86 test=89 | classes=['defective', 'good']
devices: ['/physical_device:CPU:0', '/physical_device:GPU:0']
Epoch 1/2
I0000 00:00:1786899636.848076    4083 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
13/13 ━━━━━━━━━━━━━━━━━━━━ 34s 2s/step - accuracy: 0.5320 - loss: 0.7215 - val_accuracy: 0.4419 - val_loss: 0.7137 - learning_rate: 0.0010
Epoch 2

In [14]:
# 6B. TRAIN IMPROVED - EfficientNetV2-S (stronger backbone, identical protocol)
import os, pathlib

# Define the base directory for scripts and configs relative to REPO_DIR
m2_base_dir = pathlib.Path(os.environ["REPO_DIR"]) / "ml" / "m2_condition_classification"
train_script_path = m2_base_dir / "scripts" / "train.py"
m2_yaml_path = m2_base_dir / "configs" / "m2.yaml"

!python {train_script_path} --yaml {m2_yaml_path} --only improved-effv2s

2026-08-16 17:05:09.729159: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1786899909.730728    5651 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13653 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

=== improved-effv2s | EfficientNetV2-S | m2-condition-efficientnetv2s-v0.1 ===
train=406 val=86 test=89 | classes=['defective', 'good']
devices: ['/physical_device:CPU:0', '/physical_device:GPU:0']
Epoch 1/2
2026-08-16 17:05:49.992782: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-16 17:05:50.132513: E external/local_xla/xla/stream_executor

In [16]:
import os, pathlib

# 7. EVALUATE both on the FROZEN test split + per-image latency (docs/01-prd.md 10.3)
#     Writes runs/<tag>/test_metrics.json + confusion matrix PNG/CSV.
WR = os.environ["M2_WORK_ROOT"]

# Define the base directory for scripts and configs relative to REPO_DIR
m2_base_dir = pathlib.Path(os.environ["REPO_DIR"]) / "ml" / "m2_condition_classification"
evaluate_script_path = m2_base_dir / "scripts" / "evaluate.py"

!python {evaluate_script_path} --model {WR}/runs/baseline-m3small/model.keras
!python {evaluate_script_path} --model {WR}/runs/improved-effv2s/model.keras

2026-08-16 17:19:00.819114: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1786900740.820721    9811 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13653 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786900750.930019    9851 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
model: m2-condition-mobilenetv3small-v0.1
test split: 89 images
accuracy=0.5618 | macro P=0.2809 R=0.5000 F1=0.3597
latency (per image, gpu): mean=120.95ms p50=71.08ms p95=91.88ms

per-class:
  defective    P=0.0000 R=0.0000 F1=0.0000
  good         P=0.5618 R=1.0000 F1=0.7194

saved: /kaggle/working/safresale-ml/m2/runs/baseline-m3small/test_metrics.json, test_confusion_matrix.png, test_confusion_matrix

In [18]:
# 8. SIDE-BY-SIDE METRICS (measured only - academic-integrity rule)
import json, os, pathlib
for tag in ["baseline-m3small", "improved-effv2s"]:
    p = pathlib.Path(os.environ["M2_WORK_ROOT"]) / "runs" / tag / "test_metrics.json"
    print("===", tag, "===")
    if p.exists():
        m = json.loads(p.read_text())
        print(f"  accuracy={m['accuracy']:.4f}  macro P={m['precision_macro']:.4f}  R={m['recall_macro']:.4f}  F1={m['f1_macro']:.4f}")
        lat = m["latency_per_image_ms"]
        print(f"  latency mean={lat['mean_ms']:.1f}ms p50={lat['p50_ms']:.1f}ms p95={lat['p95_ms']:.1f}ms ({lat['device']})")
        for c, s in m["per_class"].items():
            print(f"    {c:10s} P={s['precision']:.4f} R={s['recall']:.4f} F1={s['f1']:.4f}")
    else:
        print("  missing - run cell 7 first")


=== baseline-m3small ===
  accuracy=0.5618  macro P=0.2809  R=0.5000  F1=0.3597
  latency mean=121.0ms p50=71.1ms p95=91.9ms (gpu)
    defective  P=0.0000 R=0.0000 F1=0.0000
    good       P=0.5618 R=1.0000 F1=0.7194
=== improved-effv2s ===
  accuracy=0.8876  macro P=0.8859  R=0.8859  F1=0.8859
  latency mean=200.0ms p50=81.9ms p95=95.4ms (gpu)
    defective  P=0.8718 R=0.8718 F1=0.8718
    good       P=0.9000 R=0.9000 F1=0.9000


In [19]:
# 9. PACK RESULTS - download runs/ (publish weights via GitHub Releases later)
import os, pathlib, zipfile
runs = pathlib.Path(os.environ["M2_WORK_ROOT"]) / "runs"
out_zip = pathlib.Path(os.environ["M2_WORK_ROOT"]) / "m2_runs.zip"
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in runs.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(runs.parent))
print("wrote", out_zip, f"({out_zip.stat().st_size / 1e6:.1f} MB)")
if IS_COLAB:
    from google.colab import files
    files.download(str(out_zip))


wrote /kaggle/working/safresale-ml/m2/m2_runs.zip (234.2 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next steps

- Publish `m2_runs.zip` weights to a **GitHub Release**; teammates run
  `scripts/infer.py --model <downloaded>/runs/<tag>/model.keras --source <img_or_folder>`
  with no dataset required (output matches the `run-vision` condition contract, docs/04-api-contract.md).
- Fill the admin **Models** page `model_metrics` from the `test_metrics.json` files above.
- Set `VISION_PROVIDER=real` + `ML_WEIGHTS_DIR` so `run-vision` uses this model (docs/03-architecture.md §5).
- Document each dataset's URL + license in the report (docs/08-ml-plan.md §3.2).
